# Training Workbench

This notebook is the supported interactive entrypoint for training and Optuna tuning inside the Jupyter container. It reuses the callable building blocks from `train.py`, so MLflow logging, local checkpointing, MinIO dataset access, best-checkpoint promotion into `model-registry`, MLflow Model Registry registration of the best model, and study orchestration stay aligned with the script runtime.


In [5]:
from __future__ import annotations

import copy
import json
import os
import subprocess
import sys
import tempfile
from pathlib import Path

import torch

APP_ROOT = Path("/app")
if str(APP_ROOT) not in sys.path:
    sys.path.insert(0, str(APP_ROOT))

from train import (
    TrainingContext,
    TrainingResult,
    deserialize_training_result,
    ensure_supported_tune_runtime,
    find_git_repo_root,
    load_config,
    prepare_model_cache,
    resolve_hf_cache_dir,
    resolve_model_source,
    resolve_study_output_dir,
    run_optuna_study,
    sanitize_storage_path_component,
    serialize_training_context,
    set_nested_value,
    summarize_training_config,
    train_worker,
    write_json_file,
    write_yaml_file,
)

CONFIG_PATH = APP_ROOT / "config.yaml"
ACCELERATE_CONFIG_PATH = Path(os.environ.get("ACCELERATE_CONFIG_FILE", APP_ROOT / "accelerate_config.yaml"))
TRAIN_SCRIPT_PATH = Path(os.environ.get("TRAIN_SCRIPT", APP_ROOT / "train.py"))
LAST_RESULT: TrainingResult | None = None
LAST_STUDY = None


## Parameters

Edit this cell before running the execution cells below.


In [6]:
MODE = "train"
EXECUTION_MODE = "interactive"
EXPERIMENT_NAME_OVERRIDE = None
PROMOTE_BEST_CHECKPOINT_OVERRIDE = None
EXACT_RUNTIME_NUM_PROCESSES = int(os.environ.get("NUM_PROCESSES", "2"))
PREPARE_MODEL_CACHE = False

CONFIG_OVERRIDES = {
    # "data.source": "mock",
    # "training.num_epochs": 1,
    # "training.per_device_train_batch_size": 4,
}


In [7]:
def build_runtime_config() -> dict:
    cfg = copy.deepcopy(load_config(str(CONFIG_PATH)))
    for dotted_path, value in CONFIG_OVERRIDES.items():
        set_nested_value(cfg, dotted_path, value)

    if EXPERIMENT_NAME_OVERRIDE:
        cfg.setdefault("mlflow", {})
        cfg["mlflow"]["active_experiment_name"] = EXPERIMENT_NAME_OVERRIDE

    if PROMOTE_BEST_CHECKPOINT_OVERRIDE is not None:
        cfg.setdefault("model_registry", {})
        cfg["model_registry"]["promote_best_checkpoint"] = bool(PROMOTE_BEST_CHECKPOINT_OVERRIDE)

    return cfg


def resolve_best_checkpoint_registry_uri(cfg: dict, run_id: str | None) -> str | None:
    if not run_id:
        return None
    model_registry_cfg = cfg.get("model_registry", {})
    if not bool(model_registry_cfg.get("promote_best_checkpoint", True)):
        return None
    experiment_name = cfg["mlflow"].get("active_experiment_name") or cfg["mlflow"]["experiment_name"]
    bucket = str(model_registry_cfg.get("bucket") or os.environ.get("MINIO_BUCKET_MODELS", "model-registry"))
    model_name = sanitize_storage_path_component(
        str(model_registry_cfg.get("model_name") or cfg["model"]["name"])
    )
    experiment_component = sanitize_storage_path_component(experiment_name)
    run_component = sanitize_storage_path_component(run_id)
    return f"s3://{bucket}/{model_name}/experiments/{experiment_component}/runs/{run_component}/best-checkpoint"


def print_section(title: str, payload) -> None:
    print(f"\n=== {title} ===")
    if isinstance(payload, (dict, list)):
        print(json.dumps(payload, indent=2, sort_keys=True, default=str))
    else:
        print(payload)


def run_interactive_training(cfg: dict) -> TrainingResult:
    context = TrainingContext(mode="train")
    return train_worker(copy.deepcopy(cfg), context=context)


def run_exact_runtime_training(cfg: dict) -> TrainingResult:
    with tempfile.TemporaryDirectory(prefix="notebook-train-") as temp_dir:
        temp_root = Path(temp_dir)
        config_path = temp_root / "resolved_config.yaml"
        context_path = temp_root / "training_context.json"
        result_path = temp_root / "result.json"
        progress_path = temp_root / "progress.jsonl"

        context = TrainingContext(
            mode="train",
            progress_file=str(progress_path),
            result_file=str(result_path),
        )
        write_yaml_file(config_path, cfg)
        write_json_file(context_path, serialize_training_context(context))

        command = [
            sys.executable,
            "-m",
            "accelerate.commands.launch",
            "--num_processes",
            str(EXACT_RUNTIME_NUM_PROCESSES),
            "--config_file",
            str(ACCELERATE_CONFIG_PATH),
            str(TRAIN_SCRIPT_PATH),
            "--config",
            str(config_path),
            "--mode",
            "train",
            "--context-file",
            str(context_path),
        ]
        print_section("Exact-Runtime Command", " ".join(command))
        completed = subprocess.run(command, check=False)

        if not result_path.exists():
            raise RuntimeError(
                f"Exact-runtime training exited with code {completed.returncode} before writing {result_path}."
            )

        payload = json.loads(result_path.read_text(encoding="utf-8"))
        status = payload.get("status")
        if status == "complete":
            return deserialize_training_result(payload["result"])
        if status == "pruned":
            raise RuntimeError(payload.get("message", "Training run was pruned."))

        raise RuntimeError(
            f"Exact-runtime training failed with status={status!r}: "
            f"{payload.get('error_type', 'RuntimeError')}: {payload.get('error_message', 'unknown error')}"
        )


def summarize_execution_plan(cfg: dict) -> None:
    print_section("Execution Parameters", {
        "mode": MODE,
        "execution_mode": EXECUTION_MODE,
        "experiment_name_override": EXPERIMENT_NAME_OVERRIDE,
        "promote_best_checkpoint_override": PROMOTE_BEST_CHECKPOINT_OVERRIDE,
        "exact_runtime_num_processes": EXACT_RUNTIME_NUM_PROCESSES,
        "config_overrides": CONFIG_OVERRIDES,
    })
    print_section("Training Summary", summarize_training_config(cfg))
    print_section("Evaluation", cfg["evaluation"])
    print_section("Model Registry", cfg.get("model_registry", {}))
    if MODE == "tune":
        print_section("Optuna", cfg.get("optuna", {}))
    print_section("Resolved Paths", {
        "config_path": str(CONFIG_PATH),
        "train_script": str(TRAIN_SCRIPT_PATH),
        "accelerate_config_path": str(ACCELERATE_CONFIG_PATH),
        "checkpoint_dir": cfg["checkpointing"]["checkpoint_dir"],
        "hf_cache_dir": str(resolve_hf_cache_dir(cfg)),
        "model_source": resolve_model_source(cfg),
        "study_output_dir": str(resolve_study_output_dir(cfg)),
    })


## Bootstrap Checks

Run this once after the kernel starts. It verifies the container wiring expected by the notebook and by `train.py`.


In [8]:
def resolve_git_commit(repo_root: Path | None) -> str:
    if repo_root is None:
        return "unavailable"

    git_result = subprocess.run(
        ["git", "-C", str(repo_root), "rev-parse", "HEAD"],
        check=False,
        capture_output=True,
        text=True,
    )
    if git_result.returncode == 0:
        return git_result.stdout.strip() or "unavailable"

    git_dir = repo_root / ".git"
    head_path = git_dir / "HEAD"
    if not head_path.exists():
        return "unavailable"

    head_value = head_path.read_text(encoding="utf-8").strip()
    if not head_value:
        return "unavailable"
    if not head_value.startswith("ref: "):
        return head_value

    ref_name = head_value[5:].strip()
    ref_path = git_dir / ref_name
    if ref_path.exists():
        return ref_path.read_text(encoding="utf-8").strip() or "unavailable"

    packed_refs_path = git_dir / "packed-refs"
    if packed_refs_path.exists():
        for raw_line in packed_refs_path.read_text(encoding="utf-8").splitlines():
            line = raw_line.strip()
            if not line or line.startswith("#") or line.startswith("^"):
                continue
            sha, _, packed_ref = line.partition(" ")
            if packed_ref.strip() == ref_name:
                return sha.strip() or "unavailable"

    return "unavailable"


repo_root = find_git_repo_root(APP_ROOT)
required_env = [
    "MLFLOW_TRACKING_URI",
    "MLFLOW_S3_ENDPOINT_URL",
    "MINIO_ENDPOINT",
    "AWS_ACCESS_KEY_ID",
    "AWS_SECRET_ACCESS_KEY",
    "REDIS_HOST",
    "REDIS_PORT",
    "NUM_PROCESSES",
]

git_commit = resolve_git_commit(repo_root)

bootstrap_status = {
    "cwd": str(Path.cwd()),
    "python_executable": sys.executable,
    "config_exists": CONFIG_PATH.exists(),
    "train_script_exists": TRAIN_SCRIPT_PATH.exists(),
    "accelerate_config_exists": ACCELERATE_CONFIG_PATH.exists(),
    "git_repo_root": str(repo_root) if repo_root else None,
    "git_commit": git_commit,
    "gpu_available": torch.cuda.is_available(),
    "gpu_count": torch.cuda.device_count() if torch.cuda.is_available() else 0,
    "cuda_version": torch.version.cuda,
    "num_processes_env": os.environ.get("NUM_PROCESSES"),
    "missing_env": [key for key in required_env if not os.environ.get(key)],
}
print_section("Bootstrap Status", bootstrap_status)

assert CONFIG_PATH.exists(), f"Missing config file: {CONFIG_PATH}"
assert TRAIN_SCRIPT_PATH.exists(), f"Missing training script: {TRAIN_SCRIPT_PATH}"
assert repo_root is not None, "Expected /app/.git to be mounted into the Jupyter container."



=== Bootstrap Status ===
{
  "accelerate_config_exists": true,
  "config_exists": true,
  "cuda_version": "12.1",
  "cwd": "/app/notebooks",
  "git_commit": "7cbbfd6b02bd8c9650ed80aab2e46636e3490d3b",
  "git_repo_root": "/app",
  "gpu_available": true,
  "gpu_count": 2,
  "missing_env": [],
  "num_processes_env": "2",
  "python_executable": "/opt/venv/bin/python",
  "train_script_exists": true
}


## Build Runtime Config

This cell loads `/app/config.yaml`, applies notebook overrides, and shows the effective settings before you train or tune.


In [9]:
cfg = build_runtime_config()
summarize_execution_plan(cfg)

if PREPARE_MODEL_CACHE:
    cached_model_path = prepare_model_cache(cfg)
    print_section("Prepared Model Cache", cached_model_path)
else:
    print("Set PREPARE_MODEL_CACHE = True if you want to warm the Hugging Face cache before execution.")



=== Execution Parameters ===
{
  "config_overrides": {},
  "exact_runtime_num_processes": 2,
  "execution_mode": "interactive",
  "experiment_name_override": null,
  "mode": "train",
  "promote_best_checkpoint_override": null
}

=== Training Summary ===
{
  "checkpoint_dir": "/app/checkpoints/recipe-correction",
  "data_source": "mock",
  "eval_batch_size": 8,
  "gradient_accumulation_steps": 4,
  "learning_rate": 0.0003,
  "model_name": "t5-small",
  "model_source": "t5-small",
  "num_epochs": 3,
  "tracking_uri": "http://mlflow:5000",
  "train_batch_size": 8,
  "warmup_ratio": 0.06
}

=== Evaluation ===
{
  "eval_steps": 500,
  "logging_steps": 50,
  "metric_for_best_model": "eval_rougeL"
}

=== Model Registry ===
{
  "bucket": "model-registry",
  "model_name": "recipe-correction-t5",
  "promote_best_checkpoint": true
}

=== Optuna ===
{
  "direction": "maximize",
  "n_trials": 10,
  "pruner": {
    "n_startup_trials": 2,
    "n_warmup_steps": 1,
    "type": "median"
  },
  "sampler

## Interactive Training

Use this when you want to stay in-kernel for fast iteration on a single controller process while still using the real `train_worker` implementation.


In [11]:
LAST_RESULT = None
LAST_STUDY = None

if MODE != "train" or EXECUTION_MODE != "interactive":
    print("Skip this cell unless MODE='train' and EXECUTION_MODE='interactive'.")
else:
    interactive_cfg = build_runtime_config()
    LAST_RESULT = run_interactive_training(interactive_cfg)
    print_section("Interactive Training Result", {
        "run_id": LAST_RESULT.run_id,
        "best_metric_name": LAST_RESULT.best_metric_name,
        "best_metric": LAST_RESULT.best_metric,
        "best_checkpoint": LAST_RESULT.best_checkpoint,
        "final_metrics": LAST_RESULT.final_metrics,
    })



[rank 0] ----------------------------------------------------------------------------------------
[rank 0] TRAINING START
[rank 0] Initializing training worker and loading configuration.
[rank 0] Execution mode: train
[rank 0] Using mixed precision mode: no
[rank 0] Training config summary: {"checkpoint_dir": "/app/checkpoints/recipe-correction", "data_source": "mock", "eval_batch_size": 8, "gradient_accumulation_steps": 4, "learning_rate": 0.0003, "model_name": "t5-small", "model_source": "t5-small", "num_epochs": 3, "tracking_uri": "http://mlflow:5000", "train_batch_size": 8, "warmup_ratio": 0.06}


Fetching 20 files:   0%|          | 0/20 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]


[rank 0] ----------------------------------------------------------------------------------------
[rank 0] MODEL SETUP
[rank 0] Loaded tokenizer and model for /app/checkpoints/huggingface-cache/models--t5-small/snapshots/df1b051c49625cf57a3d0d8d3863ed4d13564fe4.

[rank 0] ----------------------------------------------------------------------------------------
[rank 0] DATA READY
[rank 0] Dataset summary: train_examples=200, val_examples=40, train_batches=25, val_batches=5

[rank 0] ----------------------------------------------------------------------------------------
[rank 0] OPTIMIZER SETUP
[rank 0] Optimizer/scheduler summary: steps_per_epoch=7, total_steps=21, warmup_steps=1, processes=1


2026/04/04 00:38:01 INFO mlflow.tracking.fluent: Experiment with name 'recipe-correction-t5' does not exist. Creating a new experiment.


...


[rank 0] ----------------------------------------------------------------------------------------
[rank 0] EPOCH 1 COMPLETE
[rank 0] Epoch 1 complete. Checkpoint saved to /app/checkpoints/recipe-correction/1a81d6d43122451fbfea6822b8ff556a/epoch-01. Metrics: {"epoch_time_sec": 10.0, "eval_loss": 0.5669534146785736, "eval_rouge1": 0.7064636752136751, "eval_rouge2": 0.6251984126984129, "eval_rougeL": 0.7054914529914531, "gpu_memory_allocated_gb": 8.2, "gpu_memory_reserved_gb": 9.179, "samples_per_sec": 20.0}

[rank 0] ----------------------------------------------------------------------------------------
[rank 0] BEST MODEL UPDATED
[rank 0] New best checkpoint at epoch 1: /app/checkpoints/recipe-correction/1a81d6d43122451fbfea6822b8ff556a/epoch-01 (eval_rougeL=0.7055)

[rank 0] ----------------------------------------------------------------------------------------
[rank 0] EPOCH 2/3
[rank 0] Starting epoch 2/3.
[rank 0] First training batch summary: input_ids=shape(8, 512),dtype=torch.

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


[rank 0] ----------------------------------------------------------------------------------------
[rank 0] EPOCH 2 COMPLETE
[rank 0] Epoch 2 complete. Checkpoint saved to /app/checkpoints/recipe-correction/1a81d6d43122451fbfea6822b8ff556a/epoch-02. Metrics: {"epoch_time_sec": 9.56, "eval_loss": 0.1678853750228882, "eval_rouge1": 0.9049999999999998, "eval_rouge2": 0.8940058479532162, "eval_rougeL": 0.9012499999999999, "gpu_memory_allocated_gb": 8.2, "gpu_memory_reserved_gb": 9.179, "samples_per_sec": 20.92}

[rank 0] ----------------------------------------------------------------------------------------
[rank 0] BEST MODEL UPDATED
[rank 0] New best checkpoint at epoch 2: /app/checkpoints/recipe-correction/1a81d6d43122451fbfea6822b8ff556a/epoch-02 (eval_rougeL=0.9012)

[rank 0] ----------------------------------------------------------------------------------------
[rank 0] EPOCH 3/3
[rank 0] Starting epoch 3/3.
[rank 0] First training batch summary: input_ids=shape(8, 512),dtype=torch

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


[rank 0] ----------------------------------------------------------------------------------------
[rank 0] EPOCH 3 COMPLETE
[rank 0] Epoch 3 complete. Checkpoint saved to /app/checkpoints/recipe-correction/1a81d6d43122451fbfea6822b8ff556a/epoch-03. Metrics: {"epoch_time_sec": 9.55, "eval_loss": 0.07920714020729065, "eval_rouge1": 0.9774999999999998, "eval_rouge2": 0.975, "eval_rougeL": 0.9774999999999998, "gpu_memory_allocated_gb": 8.2, "gpu_memory_reserved_gb": 9.179, "samples_per_sec": 20.94}

[rank 0] ----------------------------------------------------------------------------------------
[rank 0] BEST MODEL UPDATED
[rank 0] New best checkpoint at epoch 3: /app/checkpoints/recipe-correction/1a81d6d43122451fbfea6822b8ff556a/epoch-03 (eval_rougeL=0.9775)

RUN SUMMARY
mode                    : train
run_id                  : 1a81d6d43122451fbfea6822b8ff556a
experiment_name         : recipe-correction-t5
model_name              : t5-small
data_source             : mock
total_epochs    

## Exact-Runtime Training

Use this when you want the notebook to launch the same Accelerate CLI path that the script uses in containerized training.


In [10]:
LAST_RESULT = None
LAST_STUDY = None

if MODE != "train" or EXECUTION_MODE != "exact-runtime":
    print("Skip this cell unless MODE='train' and EXECUTION_MODE='exact-runtime'.")
else:
    exact_runtime_cfg = build_runtime_config()
    LAST_RESULT = run_exact_runtime_training(exact_runtime_cfg)
    print_section("Exact-Runtime Training Result", {
        "run_id": LAST_RESULT.run_id,
        "best_metric_name": LAST_RESULT.best_metric_name,
        "best_metric": LAST_RESULT.best_metric,
        "best_checkpoint": LAST_RESULT.best_checkpoint,
        "final_metrics": LAST_RESULT.final_metrics,
    })


Skip this cell unless MODE='train' and EXECUTION_MODE='exact-runtime'.


## Tune With Optuna

Tune mode already uses subprocess-launched Accelerate trials internally, so this cell is the exact study controller path from `train.py`.


In [ ]:
LAST_RESULT = None
LAST_STUDY = None

if MODE != "tune":
    print("Skip this cell unless MODE='tune'.")
else:
    tune_cfg = build_runtime_config()
    ensure_supported_tune_runtime()
    LAST_STUDY = run_optuna_study(copy.deepcopy(tune_cfg))
    print_section("Tune Result", {
        "study_name": LAST_STUDY.study_name,
        "best_value": LAST_STUDY.best_value,
        "best_trial_number": LAST_STUDY.best_trial.number,
        "best_params": LAST_STUDY.best_trial.params,
        "study_output_dir": str(resolve_study_output_dir(tune_cfg)),
    })


## Result Summary

Re-run this after a training or tuning cell to inspect the latest outputs, the promoted best-checkpoint location in `model-registry`, and the MLflow Model Registry name used for the best model.


In [ ]:
summary_cfg = build_runtime_config()

if LAST_RESULT is not None:
    print_section("Last Training Run", {
        "run_id": LAST_RESULT.run_id,
        "best_metric_name": LAST_RESULT.best_metric_name,
        "best_metric": LAST_RESULT.best_metric,
        "best_checkpoint": LAST_RESULT.best_checkpoint,
        "best_checkpoint_s3_uri": resolve_best_checkpoint_registry_uri(summary_cfg, LAST_RESULT.run_id),
        "mlflow_registered_model_name": (
            summary_cfg.get("model_registry", {}).get("model_name")
            if summary_cfg.get("model_registry", {}).get("log_to_mlflow_model_registry", True)
            else None
        ),
        "checkpoint_dir": summary_cfg["checkpointing"]["checkpoint_dir"],
        "model_registry": summary_cfg.get("model_registry", {}),
        "final_metrics": LAST_RESULT.final_metrics,
    })
elif LAST_STUDY is not None:
    print_section("Last Study", {
        "study_name": LAST_STUDY.study_name,
        "best_value": LAST_STUDY.best_value,
        "best_trial_number": LAST_STUDY.best_trial.number,
        "best_params": LAST_STUDY.best_trial.params,
        "study_output_dir": str(resolve_study_output_dir(summary_cfg)),
        "best_config_path": str(resolve_study_output_dir(summary_cfg) / "best_config.yaml"),
        "trial_summary_path": str(resolve_study_output_dir(summary_cfg) / "trial_summary.json"),
        "mlflow_registered_model_name": (
            summary_cfg.get("model_registry", {}).get("model_name")
            if summary_cfg.get("model_registry", {}).get("log_to_mlflow_model_registry", True)
            else None
        ),
        "model_registry": summary_cfg.get("model_registry", {}),
    })
else:
    print("No training or tuning result has been captured in this kernel yet.")
